<a href="https://colab.research.google.com/github/Ape108/FundamentalAnalysisGPT/blob/main/training_code/Milestone_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Config & Setup

In [110]:
!pip install -U "datasets<3.0.0"

### Imports

In [111]:
import re # regular expressions
import math # math utilities
import random # randomly seeding
from collections import Counter # count token frequency
from collections import defaultdict
import string
import datasets

import torch # PyTorch tensor library
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader # dataset and batching

# parallelizing for faster training
import os
import itertools

torch.set_float32_matmul_precision('high')

In [112]:
# seeding random numbers for reproducibility
random.seed(42)
torch.manual_seed(42)

In [113]:
device = "cuda" if torch.cuda.is_available() else "cpu"  # choose device
print("Using device:", device)

Using device: cuda


In [114]:
CONFIG = {
    "vocab_size": 0,
    "context_length": 256,
    "emb_dim": 256,
    "n_heads": 8,
    "n_layers": 4,
    "drop_rate": 0.1,
    "qkv_bias": False,
    "batch_size": 256,
    "learning_rate": 5e-4,
    "max_steps": 3000
}

print("Config:", CONFIG)

Config: {'vocab_size': 0, 'context_length': 256, 'emb_dim': 256, 'n_heads': 8, 'n_layers': 4, 'drop_rate': 0.1, 'qkv_bias': False, 'batch_size': 256, 'learning_rate': 0.0005, 'max_steps': 3000}


# Data Pipeline

In [115]:
year_1993_training_dataset, year_1993_test_dataset = datasets.load_dataset(
    "eloukas/edgar-corpus",
    "year_1993",
    split=["train","test"],
    trust_remote_code=True
)

In [116]:
class SECRegexTokenizer:
    def __init__(self):
        self.vocab = {}
        self.inv_vocab = {}

    def _tokenize_text(self, text):
        # Base regex splitting logic
        tokens = re.split(r"([,.:;?_!/\"()'\-]|\s+)", text)
        return [t.strip() for t in tokens if t.strip()]

    def build_vocab(self, corpus_texts):
        print("Building vocabulary...")

        all_tokens = []
        for text in corpus_texts:
            all_tokens.extend(self._tokenize_text(text))

        unique = sorted(set(all_tokens))
        self.vocab = {tok: i for i, tok in enumerate(unique)}

        special_tokens = ['<|unk|>', '<|endoftext|>', '[BOS]', '[EOS]', '[PAD]']
        for tok in special_tokens:
            if tok not in self.vocab:
                new_token_id = len(self.vocab)
                self.vocab[tok] = new_token_id

        self.inverse_vocab = {i: tok for tok, i in self.vocab.items()}
        print(f"Vocabulary built! Size: {len(self.vocab)}")


    def encode(self, text, unk_token='<|unk|>'):
        toks = self._tokenize_text(text) # Tokenize the new input text
        unk_id = self.vocab[unk_token] # Get the ID for the unknown tokens
        ids = [self.vocab.get(t, unk_id) for t in toks] # We use vocab.get to fall back to unk_id
        return ids, toks

    def decode(self, ids):
        toks = [self.inverse_vocab[i] for i in ids] # map each ID to its token
        text = ' '.join(toks) # join tokens with spaces
        text = re.sub(r"\s+([,.:;?_!/\"()'])", r"\1", text)
        text = re.sub(r"\s+\-", "-", text)
        return text

class EDGARDataset(torch.utils.data.Dataset):
    def __init__(self, data_tokens, context_length):
        self.data = data_tokens
        self.context_length = context_length

    def __len__(self):
        return len(self.data) - self.context_length

    def __getitem__(self, idx):
        # Returns (Context, Target) pairs
        chunk = self.data[idx : idx + self.context_length + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y

# Model Architecture

In [117]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim: int, eps: float = 1e-5):
        super().__init__()  # initialize base class

        self.eps = eps  # numerical stability term

        # Learnable parameters: scale (gamma) and shift (beta)
        self.gamma = nn.Parameter(torch.ones(emb_dim))  # [D]
        self.beta = nn.Parameter(torch.zeros(emb_dim))  # [D]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]

        # Compute mean over the feature dimension D
        mean = x.mean(dim=-1, keepdim=True)  # [B, T, 1]

        # Compute variance over the feature dimension D
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # [B, T, 1]

        # Normalize
        x_hat = (x - mean) / torch.sqrt(var + self.eps)  # [B, T, D]

        # Scale and shift (broadcast gamma/beta over B and T)
        out = self.gamma * x_hat + self.beta  # [B, T, D]

        return out

In [118]:
class FeedForward(nn.Module):
    def __init__(self, emb_dim: int, drop_rate: float):
        super().__init__()  # initialize

        # Two-layer MLP with GELU in between
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),  # expand features
            nn.GELU(),                        # nonlinearity
            nn.Linear(4 * emb_dim, emb_dim),  # project back to emb_dim
            nn.Dropout(drop_rate)             # dropout for regularization
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        return self.net(x)  # [B, T, D]

In [119]:
class MultiHeadCausalSelfAttention(nn.Module):
    # Multi-head causal self-attention (scaled dot-product)
    def __init__(self, emb_dim: int, num_heads: int, context_length: int, drop_rate: float, qkv_bias: bool):
        super().__init__()  # init

        assert emb_dim % num_heads == 0  # ensure heads divide embedding dim

        self.emb_dim = emb_dim  # embedding dimension D
        self.num_heads = num_heads  # number of heads H
        self.head_dim = emb_dim // num_heads  # per-head dim

        # Linear projections for Q, K, V (each produces D features)
        self.Wq = nn.Linear(emb_dim, emb_dim, bias=qkv_bias)
        self.Wk = nn.Linear(emb_dim, emb_dim, bias=qkv_bias)
        self.Wv = nn.Linear(emb_dim, emb_dim, bias=qkv_bias)

        # Output projection back to emb_dim
        self.out_proj = nn.Linear(emb_dim, emb_dim, bias=True)

        # Dropout on attention weights
        self.attn_drop = nn.Dropout(drop_rate)

        # Register a causal mask as a non-parameter buffer
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        B, T, D = x.shape  # unpack

        # Project to Q, K, V: [B, T, D]
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Reshape into heads: [B, T, D] -> [B, H, T, head_dim]
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # Scores: [B, H, T, T]
        scores = Q @ K.transpose(-2, -1)

        # Scale scores for stability
        scores = scores / math.sqrt(self.head_dim)

        # Apply causal mask (slice to current T)
        mask = self.mask[:T, :T]
        scores = scores.masked_fill(mask, -torch.inf)

        # Softmax over last dim to get attention weights
        weights = torch.softmax(scores, dim=-1)

        # Apply dropout to attention weights
        weights = self.attn_drop(weights)

        # Context per head: [B, H, T, head_dim]
        context = weights @ V

        # Recombine heads: [B, H, T, head_dim] -> [B, T, D]
        context = context.transpose(1, 2).contiguous().view(B, T, D)

        # Final projection: [B, T, D]
        out = self.out_proj(context)

        return out

In [120]:
class TransformerBlock(nn.Module):
    # GPT-2 style Pre-LN transformer block: (Attn + FFN) with residual connections
    def __init__(self, cfg):
        super().__init__()  # init

        D = cfg["emb_dim"]  # embedding dim

        # Pre-LN layers
        self.ln1 = LayerNorm(D)
        self.ln2 = LayerNorm(D)

        # Causal multi-head attention
        self.attn = MultiHeadCausalSelfAttention(
            emb_dim=D,
            num_heads=cfg["n_heads"],
            context_length=cfg["context_length"],
            drop_rate=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )

        # Feed-forward network
        self.ff = FeedForward(D, cfg["drop_rate"])

        # Dropout on residual branches (common in GPT-style)
        self.resid_drop = nn.Dropout(cfg["drop_rate"])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]

        # Attention block (Pre-LN) + residual
        x = x + self.resid_drop(self.attn(self.ln1(x)))

        # Feed-forward block (Pre-LN) + residual
        x = x + self.resid_drop(self.ff(self.ln2(x)))

        return x

In [121]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()  # init

        self.cfg = cfg  # store cfg

        # Embeddings
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Transformer blocks
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # Final normalization
        self.final_ln = LayerNorm(cfg["emb_dim"])

        # Output head (logits over vocab)
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        # idx: [B, T]
        B, T = idx.shape  # unpack

        # Token embeddings: [B, T, D]
        tok = self.tok_emb(idx)

        # Positional embeddings: [T, D]
        pos_ids = torch.arange(T, device=idx.device)
        pos = self.pos_emb(pos_ids)

        # Combine + dropout: [B, T, D]
        x = self.drop_emb(tok + pos)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x)

        # Final norm
        x = self.final_ln(x)

        # Output logits: [B, T, V]
        logits = self.out_head(x)

        return logits

# Execution

In [122]:
def estimate_loss(model, val_dataloader, eval_batches=50):
    model.eval()
    batch_losses = []
    with torch.no_grad():
        # Add autocast for fast evaluation
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            for i, (X, y) in enumerate(val_dataloader):
                if i >= eval_batches: # STOP after 50 batches
                    break
                X, y = X.to(device), y.to(device)
                logits = model(X).view(-1, model.cfg["vocab_size"])
                loss = torch.nn.functional.cross_entropy(logits, y.view(-1))
                batch_losses.append(loss.item())
    model.train()
    return sum(batch_losses) / len(batch_losses) if batch_losses else 0


def generate_text(model, starting_tokens, max_new_tokens):
    model.eval()
    for _ in range(max_new_tokens):
        tokens_cond = starting_tokens[:, -model.cfg["context_length"]:]
        with torch.no_grad():
            logits = model(tokens_cond)
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        starting_tokens = torch.cat([starting_tokens, next_id], dim=1)
    return starting_tokens


def train(num_epochs):
    # 1. Initialize Tokenizer & Build Vocab
    tokenizer = SECRegexTokenizer()
    train_texts = [feature['section_1'] for feature in year_1993_training_dataset]
    tokenizer.build_vocab(train_texts)

    # 2. Update Model Config
    CONFIG["vocab_size"] = len(tokenizer.vocab)
    print(f"Updated CONFIG: {CONFIG}")

    # 3. Flatten and Tokenize Data (PARALLELIZED)
    print("Tokenizing data using multiprocessing...")
    eos_id = tokenizer.vocab['<|endoftext|>']

    def process_batch(examples):
        batch_ids = []
        for text in examples['section_1']:
            ids, _ = tokenizer.encode(text)
            batch_ids.append(ids + [eos_id])
        return {"flat_ids": batch_ids}

    # Map across all available CPU cores
    tokenized_train = year_1993_training_dataset.map(
        process_batch,
        batched=True,
        num_proc=os.cpu_count(),
        desc="Tokenizing Train Data"
    )
    all_train_tokens = list(itertools.chain.from_iterable(tokenized_train["flat_ids"]))

    tokenized_val = year_1993_test_dataset.map(
        process_batch,
        batched=True,
        num_proc=os.cpu_count(),
        desc="Tokenizing Val Data"
    )
    all_val_tokens = list(itertools.chain.from_iterable(tokenized_val["flat_ids"]))


    cores = min(8, os.cpu_count())
    print(f"Spinning up {cores} DataLoader workers...")

    # 4. Initialize Dataset & Dataloaders
    train_dataset = EDGARDataset(all_train_tokens, CONFIG["context_length"])
    train_dataloader = torch.utils.data.DataLoader(train_dataset,
                                                   batch_size=CONFIG["batch_size"],
                                                   shuffle=True,
                                                   num_workers=cores,          # Fetch data using 4 background CPU cores
                                                   pin_memory=True         # Speeds up CPU-to-GPU transfer
    )
    val_dataset = EDGARDataset(all_val_tokens, CONFIG["context_length"])
    val_dataloader = torch.utils.data.DataLoader(val_dataset,
                                                 batch_size=CONFIG["batch_size"],
                                                 shuffle=False,
                                                 num_workers=cores,
                                                 pin_memory=True
    )

    # 5. Initialize Model & Optimizer
    model = GPTModel(CONFIG).to(device)

    # Compile the model for the H100 Architecture
    print("Compiling model for H100 Arhcitecture...")
    model = torch.compile(model)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"])

    # 6. Training Loop
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):

        for step, (X, y) in enumerate(train_dataloader):

            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)

            optimizer.zero_grad()

            # Automatic Mixed Precision (bfloat16 for H100)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(X).view(-1, CONFIG["vocab_size"])
                loss = torch.nn.functional.cross_entropy(logits, y.view(-1))

            loss.backward()
            optimizer.step()

            if step % 250 == 0:
                val_loss = estimate_loss(model, val_dataloader)
                val_losses.append(val_loss)
                train_loss = loss.item()
                train_losses.append(train_loss)
                print(f"Epoch {epoch} | Step {step} | Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f}")

            if step == 1500:
                break # Stop early manually

    return model, tokenizer, train_losses, val_losses

if __name__ == "__main__":
    model, tokenizer, train_losses, val_losses = train(num_epochs=1)
    print("Pipeline run complete!")

Building vocabulary...
Vocabulary built! Size: 47319
Updated CONFIG: {'vocab_size': 47319, 'context_length': 256, 'emb_dim': 256, 'n_heads': 8, 'n_layers': 4, 'drop_rate': 0.1, 'qkv_bias': False, 'batch_size': 256, 'learning_rate': 0.0005, 'max_steps': 3000}
Tokenizing data using multiprocessing...


Tokenizing Train Data (num_proc=26):   0%|          | 0/1060 [00:00<?, ? examples/s]

Tokenizing Val Data (num_proc=26):   0%|          | 0/133 [00:00<?, ? examples/s]

Spinning up 8 DataLoader workers...
Compiling model for H100 Arhcitecture...
Epoch 0 | Step 0 | Train loss: 10.9169 | Val loss: 10.6927
Epoch 0 | Step 250 | Train loss: 5.5450 | Val loss: 5.5948
Epoch 0 | Step 500 | Train loss: 4.8662 | Val loss: 5.0498
Epoch 0 | Step 750 | Train loss: 4.4296 | Val loss: 4.7347
Epoch 0 | Step 1000 | Train loss: 4.0975 | Val loss: 4.4906
Epoch 0 | Step 1250 | Train loss: 3.9415 | Val loss: 4.3039
Epoch 0 | Step 1500 | Train loss: 3.7574 | Val loss: 4.1927
Pipeline run complete!


In [126]:
# 1. Start with your text prompt
prompt = "The Registrant has"

# 2. Encode the text into integer IDs
input_ids, _ = tokenizer.encode(prompt)

# 3. Convert to a PyTorch tensor, add a batch dimension [1, T], and move to device
input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

# 4. Generate the new tokens!
print("Generating text...")
output_tensor = generate_text(model, input_tensor, max_new_tokens=40)

# 5. Extract the 1D list of IDs from the batch and decode back to English
output_ids = output_tensor[0].tolist()
generated_text = tokenizer.decode(output_ids)

print("\n--- Final Output ---")
print(generated_text)

Generating text...

--- Final Output ---
The Registrant has a number of patents and trademarks, including the Registrant, as well as the Registrant' s trademarks and trademarks. The Registrant' s business is highly competitive. The Registrant' s principal competitors are compete with


# Works Cited

1. https://www.youtube.com/watch?v=kCc8FmEb1nY

2. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017). "Attention is all you need." Advances in neural information processing systems, 30.